In [ ]:
import os
import argparse
import subprocess
import random

def create_mix():
    parser = argparse.ArgumentParser(description="Скрипт для создания музыкального микса")
    parser.add_argument("-s", "--source", required=True, help="Путь к папке с треками")
    parser.add_argument("-d", "--destination", help="Имя выходного файла")
    parser.add_argument("-c", "--count", type=int, help="Количество файлов для обработки")
    parser.add_argument("-f", "--frame", type=int, default=10, help="Длительность фрагмента в секундах")
    parser.add_argument("-l", "--log", action="store_true", help="Выводить лог процесса")
    parser.add_argument("-e", "--extended", action="store_true", help="Добавить эффекты затухания (fade)")

    args = parser.parse_args()

    if not os.path.exists(args.source):
        print("Ошибка: Папка с музыкой не найдена.")
        return

    files = [f for f in os.listdir(args.source) if f.endswith('.mp3')]
    
    if args.count and args.count < len(files):
        files = files[:args.count]

    dest = args.destination if args.destination else os.path.join(args.source, "mix.mp3")
    
    temp_files = []

    print("--- Начинаю обработку файлов ---")

    for i, filename in enumerate(files, 1):
        input_path = os.path.join(args.source, filename)
        temp_output = os.path.join(args.source, f"temp_{i}.mp3")
        temp_files.append(temp_output)

        if args.log:
            print(f"--- processing file {i}: {filename}")

        start_time = random.randint(10, 40)
        
        cmd = [
            'ffmpeg', '-y', '-ss', str(start_time), '-t', str(args.frame),
            '-i', input_path
        ]

        if args.extended:
            cmd += ['-af', f'afade=t=in:ss=0:d=1,afade=t=out:st={args.frame-1}:d=1']

        cmd += [temp_output]

        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    list_file = "list.txt"
    with open(list_file, "w") as f:
        for temp in temp_files:
            f.write(f"file '{temp}'\n")

    concat_cmd = [
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', list_file,
        '-c', 'copy', dest
    ]
    subprocess.run(concat_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    for temp in temp_files:
        os.remove(temp)
    os.remove(list_file)

    print(f"--- Готово! Микс сохранен: {dest}")

if __name__ == "__main__":
    create_mix()